## Feature Engineering

In [31]:
import pandas as pd
import numpy as np

In [32]:
df = pd.read_csv("../data/Clean_dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [33]:
print("Original Columns:")
print(df.columns.tolist())

print("\nNumber of Features:", len(df.columns))

Original Columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Number of Features: 20


In [34]:
df["TenureYears"] = df["tenure"] / 12

df[["tenure", "TenureYears"]].head()

,tenure,TenureYears
0,1,0.083333
1,34,2.833333
2,2,0.166667
3,45,3.750000
4,2,0.166667


In [35]:
df["IsNewCustomer"] = (df["tenure"] <= 12).astype(int)

df[["tenure", "IsNewCustomer"]].head()

,tenure,IsNewCustomer
0,1,1
1,34,0
2,2,1
3,45,0
4,2,1


In [36]:
df["IsLongTermCustomer"] = (df["tenure"] >= 48).astype(int)

df[["tenure", "IsLongTermCustomer"]].head()

,tenure,IsLongTermCustomer
0,1,0
1,34,0
2,2,0
3,45,0
4,2,0


In [37]:
service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

In [38]:
df["ServiceCount"] = (
    df[service_columns] == "Yes"
).sum(axis=1)

df[["ServiceCount"] + service_columns].head()

,ServiceCount,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,1,No,Yes,No,No,No,No
1,2,Yes,No,Yes,No,No,No
2,2,Yes,Yes,No,No,No,No
3,3,Yes,No,Yes,Yes,No,No
4,0,No,No,No,No,No,No


In [39]:
df["ExpectedTotalCharges"] = (
    df["MonthlyCharges"] * df["tenure"]
)

df[
    ["MonthlyCharges", "tenure", "TotalCharges", "ExpectedTotalCharges"]
].head()

,MonthlyCharges,tenure,TotalCharges,ExpectedTotalCharges
0,29.85,1,29.85,29.85
1,56.95,34,1889.50,1936.30
2,53.85,2,108.15,107.70
3,42.30,45,1840.75,1903.50
4,70.70,2,151.65,141.40


In [40]:
df["ChargeDifference"] = (
    df["TotalCharges"] - df["ExpectedTotalCharges"]
)

df[
    ["TotalCharges", "ExpectedTotalCharges", "ChargeDifference"]
].head()

,TotalCharges,ExpectedTotalCharges,ChargeDifference
0,29.85,29.85,0.00
1,1889.50,1936.30,-46.80
2,108.15,107.70,0.45
3,1840.75,1903.50,-62.75
4,151.65,141.40,10.25


In [41]:
df["AverageMonthlyCharge"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

df[
    ["tenure", "MonthlyCharges", "TotalCharges", "AverageMonthlyCharge"]
].head()

,tenure,MonthlyCharges,TotalCharges,AverageMonthlyCharge
0,1,29.85,29.85,29.850000
1,34,56.95,1889.50,55.573529
2,2,53.85,108.15,54.075000
3,45,42.30,1840.75,40.905556
4,2,70.70,151.65,75.825000


In [42]:
df["ContractTenure"] = (
    df["Contract"].astype(str)
    + "_" +
    pd.cut(
        df["tenure"],
        bins=[-1, 12, 24, 48, np.inf],
        labels=["New", "ShortTerm", "MidTerm", "LongTerm"]
    ).astype(str)
)

df[["Contract", "tenure", "ContractTenure"]].head()

,Contract,tenure,ContractTenure
0,Month-to-month,1,Month-to-month_New
1,One year,34,One year_MidTerm
2,Month-to-month,2,Month-to-month_New
3,One year,45,One year_MidTerm
4,Month-to-month,2,Month-to-month_New


In [43]:
new_features = [
    "TenureYears",
    "IsNewCustomer",
    "IsLongTermCustomer",
    "ServiceCount",
    "ExpectedTotalCharges",
    "ChargeDifference",
    "AverageMonthlyCharge",
    "ContractTenure"
]

df[new_features].head()

,TenureYears,IsNewCustomer,IsLongTermCustomer,ServiceCount,ExpectedTotalCharges,ChargeDifference,AverageMonthlyCharge,ContractTenure
0,0.083333,1,0,1,29.85,0.00,29.850000,Month-to-month_New
1,2.833333,0,0,2,1936.30,-46.80,55.573529,One year_MidTerm
2,0.166667,1,0,2,107.70,0.45,54.075000,Month-to-month_New
3,3.750000,0,0,3,1903.50,-62.75,40.905556,One year_MidTerm
4,0.166667,1,0,0,141.40,10.25,75.825000,Month-to-month_New


In [44]:
df[new_features].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
TenureYears,7043.0,NaN,NaN,NaN,2.697596,2.046623,0.0,0.75,2.416667,4.583333,6.0
IsNewCustomer,7043.0,NaN,NaN,NaN,0.310379,0.462682,0.0,0.0,0.0,1.0,1.0
IsLongTermCustomer,7043.0,NaN,NaN,NaN,0.326991,0.469147,0.0,0.0,0.0,1.0,1.0
ServiceCount,7043.0,NaN,NaN,NaN,2.03791,1.847682,0.0,0.0,2.0,3.0,6.0
ExpectedTotalCharges,7043.0,NaN,NaN,NaN,2279.58135,2264.729447,0.0,394.0,1393.6,3786.1,8550.0
ChargeDifference,7043.0,NaN,NaN,NaN,0.152953,67.202778,-370.85,-28.6,0.0,28.5,373.25
AverageMonthlyCharge,7043.0,NaN,NaN,NaN,64.762906,30.189796,13.775,35.935156,70.3375,90.174158,121.4
ContractTenure,7043,12,Month-to-month_New,1994,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
print("Missing values in new features:")
print(df[new_features].isnull().sum())

Missing values in new features:
TenureYears             0
IsNewCustomer           0
IsLongTermCustomer      0
ServiceCount            0
ExpectedTotalCharges    0
ChargeDifference        0
AverageMonthlyCharge    0
ContractTenure          0
dtype: int64


In [46]:
numeric_new_features = [
    "TenureYears",
    "ServiceCount",
    "ExpectedTotalCharges",
    "ChargeDifference",
    "AverageMonthlyCharge"
]

print(
    "Infinite values:",
    np.isinf(df[numeric_new_features]).sum().sum()
)

Infinite values: 0


In [48]:
print("Shape after Feature Engineering:", df.shape)

Shape after Feature Engineering: (7043, 28)


In [49]:
print("All columns after Feature Engineering:")
for i, column in enumerate(df.columns, start=1):
    print(i, column)

All columns after Feature Engineering:
1 gender
2 SeniorCitizen
3 Partner
4 Dependents
5 tenure
6 PhoneService
7 MultipleLines
8 InternetService
9 OnlineSecurity
10 OnlineBackup
11 DeviceProtection
12 TechSupport
13 StreamingTV
14 StreamingMovies
15 Contract
16 PaperlessBilling
17 PaymentMethod
18 MonthlyCharges
19 TotalCharges
20 Churn
21 TenureYears
22 IsNewCustomer
23 IsLongTermCustomer
24 ServiceCount
25 ExpectedTotalCharges
26 ChargeDifference
27 AverageMonthlyCharge
28 ContractTenure


In [50]:
feature_engineered_dataset = df.copy()

feature_engineered_dataset.to_csv(
    "../data/Feature_engineered_dataset.csv",
    index=False
)

print("Feature engineered dataset saved successfully!")

Feature engineered dataset saved successfully!


In [53]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TotalCharges,Churn,TenureYears,IsNewCustomer,IsLongTermCustomer,ServiceCount,ExpectedTotalCharges,ChargeDifference,AverageMonthlyCharge,ContractTenure
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,29.85,No,0.083333,1,0,1,29.85,0.00,29.850000,Month-to-month_New
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,1889.50,No,2.833333,0,0,2,1936.30,-46.80,55.573529,One year_MidTerm
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,108.15,Yes,0.166667,1,0,2,107.70,0.45,54.075000,Month-to-month_New
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,1840.75,No,3.750000,0,0,3,1903.50,-62.75,40.905556,One year_MidTerm
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,151.65,Yes,0.166667,1,0,0,141.40,10.25,75.825000,Month-to-month_New
